[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/simon-clematide/colab-notebooks-for-teaching/blob/main/l08_transformer_companion.ipynb)

# L08 Companion: The Transformer Encoder

An encoder repeatedly **exchanges information between tokens** and **processes each token locally**. This workbook connects that idea to formulas, small examples, and code. It uses row vectors and puts the weight matrix second: $Q=XW^Q$.

| Route | Sections | Question |
|---|---|---|
| **Main route** | N1 → N2 → N4 → N5 → N6 | How does an encoder build contextual representations? |
| **Deeper investigation** | N3 | Why divide scores by $\sqrt{d_k}$? |
| **Synthesis experiment** | N7 | Can training recover order without position information? |
| **Extensions** | N8, N9 | How do masks, cross-attention, and subwords fit in? |

**Predict → run → explain.** Write a prediction before executing each experiment. Expand explanations when needed and solutions after trying the question. Implementation cells remain executable; their source can be folded in notebook viewers that support it. Markdown disclosures do not hide separate code cells.

Run the setup below first. For the main route, run its sections in order; N3 may be skipped. N7 needs N4–N6, N8 needs N4, and N9 needs only setup. **Run all** also works in document order. Experiments use CPU; runtime varies by machine.

<details>
<summary>Setup and dependency notes</summary>

The required packages are `torch` and `matplotlib`. In a notebook environment missing them, install them with `%pip install torch matplotlib`, then restart the kernel if needed. The optional pretrained-tokenizer example in N9 additionally needs `transformers` and network access. The toy BPE example runs offline.

The examples use fixed seeds for reproducibility; training results can still vary across software and hardware.

</details>

In [ ]:
# Dependencies: PyTorch and matplotlib
import math
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42)
torch.set_printoptions(precision=3, sci_mode=False)
print(f"PyTorch version: {torch.__version__}")

## N1 — One Worked Step

**Attention scores keys, normalizes scores, and mixes values.** All three projections come from the same input; token 2 supplies the query in this example.

**Predict:** Given $q_2=(1,1)$ and keys $(1,0),(1,2),(0,4)$, which token receives the largest weight? Compute the three dot products before running.

<details>
<summary>Connection to the lecture and notation</summary>

Slides: “Your Turn: One Worked Step” and “All Queries at Once.” Tokens are rows. We omit scaling here to reproduce the worked example; N3 investigates scaling. The matrices below are deliberately chosen, not trained.

</details>

In [ ]:
X = torch.tensor([[1., 0.], [1., 1.], [0., 2.]])     # n=3 tokens of width d=2, one per row
W_Q = torch.eye(2)                                   # identity
W_K = torch.diag(torch.tensor([1., 2.]))             # stretches the second axis
W_V = torch.tensor([[0., 1.], [1., 0.]])             # swaps the axes

Q, K, V = X @ W_Q, X @ W_K, X @ W_V
print("Q =\n", Q, "\nK =\n", K, "\nV =\n", V)

# One query, written out: score, normalize, mix
q_2 = Q[1]
scores = torch.stack([q_2 @ K[j] for j in range(3)])
weights = F.softmax(scores, dim=-1)
y_2 = sum(weights[j] * V[j] for j in range(3))
print("scores :", scores)
print("weights:", weights)
print("y_2    :", y_2)
print("raw dot products with x_2:", X[1] @ X.T)

The same computation for all queries at once needs only matrix products. Row $i$ of $S$ holds query $i$ compared with all keys.

$$S=QK^{\top},\qquad P=\operatorname{softmax}_{\text{row}}(S),\qquad Y=PV$$

In [ ]:
S = Q @ K.T
P = F.softmax(S, dim=-1)        # dim=-1: normalize each row
Y = P @ V
print("S =\n", S, "\nP =\n", P, "\nY =\n", Y)
print("row sums of P   :", P.sum(dim=1))
print("column sums of P:", P.sum(dim=0))
print("matches the loop:", torch.allclose(Y[1], y_2))

**Check yourself.**

1. Which matrices in N1 would be updated during training? Which quantities are computed again for each new input?
2. Replace the three-token input with 300 tokens of the same width. Do the learned projection matrices need to grow? What changes in the matrix computation?

<details>
<summary>Solution: shared parameters versus input-dependent quantities</summary>

- **Training changes the projection matrices** $W^Q,W^K,W^V$. In this toy example they were chosen by hand. In a trained model, they remain fixed during ordinary inference.
- **Each input produces new intermediate quantities:** $Q,K,V,S,P,Y$. The input matrix $X$ also has one row per token.
- **More tokens do not require larger projection matrices.** The same projections apply to every row. With 300 tokens, $X,Q,K,V,Y$ have 300 rows, and $S,P$ have shape $300\times300$. The matrix expressions in the all-queries cell stay the same; the earlier hand-written loop is specific to three tokens.

</details>

## N2 — Keys Choose Access; Values Supply Content

**Changing a value changes the content; changing a key can redistribute access.** Predict both interventions before running them.

| Intervention for query 2 | Do attention weights change? | Does the output change? Why? |
|---|---|---|
| Set only $v_3$ to zero | … | … |
| Reset; set only $k_3$ to zero | … | … |

<details>
<summary>Why intervene on projected vectors?</summary>

Slide: “Explain to a Partner: Keys and Values.” These interventions change $K$ or $V$ directly and leave everything else fixed. Editing the original input token could change its query, key, and value together, so it would not isolate their roles.

</details>

In [ ]:
def attend_row(q, K, V):
    w = F.softmax(q @ K.T, dim=-1)
    return w, w @ V

w0, y0 = attend_row(Q[1], K, V)
print("original      weights", w0, "output", y0)

V_new = V.clone(); V_new[2] = 0.                # change only v_3
w1, y1 = attend_row(Q[1], K, V_new)
print("v_3 = (0,0)   weights", w1, "output", y1)

K_new = K.clone(); K_new[2] = 0.                # change only k_3
w2, y2 = attend_row(Q[1], K_new, V)
print("k_3 = (0,0)   weights", w2, "output", y2)

**Check yourself.**

1. When only $v_3$ is set to zero, why do the attention weights stay the same while the output changes?
2. When only $k_3$ is set to zero, its score becomes zero. Why does token 3 still receive some weight?
3. Why do tokens 1 and 2 gain weight even though their keys have not changed?

<details>
<summary>Solution: content changes versus redistribution of access</summary>

- **Values do not determine the weights.** The weights depend on the query and keys, which were held fixed. Removing $v_3$ therefore leaves the weights unchanged but removes its contribution to the weighted sum.
- **A zero score still has positive softmax weight.** Its numerator is $\exp(0)=1$, not zero. To exclude a position in this computation, mask its score with $-\infty$ before softmax.
- **All weights share a denominator.** The third score falls from 4 to 0, reducing the sum of exponentiated scores. The unchanged numerators for tokens 1 and 2 therefore become larger fractions of the total. Changing one key can redistribute the entire row of weights.

</details>

## N3 — Why Scale? · Deeper Investigation

**Scaling controls score spread without changing key ranking.** Under independent, zero-mean, unit-variance components, dot-product variance grows with $d_k$.

**Scaling acts like a temperature adjustment.** Wider query and key vectors tend to produce more spread-out dot products. Dividing by $\sqrt{d_k}$ compensates for that growth, so attention does not become sharper simply because the vectors are wider.

<details>
<summary>Connection to softmax temperature</summary>

Writing $P=\operatorname{softmax}_{\mathrm{row}}(S/T)$ with raw scores $S=QK^{\top}$, attention scaling corresponds to $T=\sqrt{d_k}$. Increasing a positive temperature softens the weights without changing key ranking. Here the temperature is set by head width to calibrate the score scale; uniform attention is not the goal.

</details>

**Predict before running the two experiments:**

1. As head width grows, should raw score variance grow or stay roughly constant? What about variance after division by $\sqrt{d_k}$?
2. For the unscaled scores, should the largest attention weight move closer to 1? Should the softmax become more or less sensitive to small changes in its scores?

<details>
<summary>What the gradient experiment measures</summary>

Slide: “Scaling Keeps the Softmax Trainable.” The second experiment measures the sensitivity of one attention weight, $p_0$, to the scores supplied to softmax: $\|\partial p_0/\partial s\|$. Large score differences often concentrate probability and reduce this sensitivity. This is an illustration of softmax saturation, not a measurement of the gradients of a trained model's loss with respect to its parameters.

The scaled and unscaled conditions reuse identical sampled raw scores. The measured gradients are with respect to each condition's softmax input, not the raw scores before scaling.

</details>

In [ ]:
torch.manual_seed(0)
print(f"{'d_k':>6} {'var(score)':>12} {'var(scaled)':>12}")
for d_k in [4, 16, 64, 256, 1024]:
    q, k = torch.randn(20000, d_k), torch.randn(20000, d_k)
    s = (q * k).sum(dim=1)
    print(f"{d_k:6d} {s.var().item():12.1f} {(s / math.sqrt(d_k)).var().item():12.2f}")

In [ ]:
def softmax_stats(raw_scores, scale=1.0):
    scores = (raw_scores / scale).detach().requires_grad_(True)
    p = F.softmax(scores, dim=-1)
    grad, = torch.autograd.grad(p[:, 0].sum(), scores)
    return p.max(dim=-1).values.mean().item(), grad.norm(dim=-1).mean().item()

torch.manual_seed(0)
d_ks = [4, 16, 64, 256, 1024]
stats = {False: [], True: []}
for d_k in d_ks:
    q = torch.randn(500, 1, d_k)
    k = torch.randn(500, 10, d_k)
    raw_scores = (q @ k.transpose(1, 2)).squeeze(1)
    stats[False].append(softmax_stats(raw_scores))
    stats[True].append(softmax_stats(raw_scores, math.sqrt(d_k)))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for sc, label in [(False, "unscaled"), (True, r"divided by $\sqrt{d_k}$")]:
    axes[0].semilogx(d_ks, [a for a, _ in stats[sc]], "o-", label=label)
    axes[1].loglog(d_ks, [b for _, b in stats[sc]], "o-", label=label)
axes[0].set(xlabel="$d_k$", ylabel="mean largest weight", title="Attention concentration")
axes[1].set(xlabel="$d_k$", ylabel="mean norm of d p₀ / d scores", title="Sensitivity of one weight")
for ax in axes:
    ax.grid(alpha=.3)
    ax.legend()
plt.tight_layout()
plt.show()

**Check yourself.**

1. Compare scores $(1,3,4)$ with those scores divided by $\sqrt{2}$. Does the key with the largest weight change? Does its share of the total change?
2. In the plots, which condition becomes more concentrated as head width grows? How does this relate to the sensitivity of one attention weight to the scores?

<details>
<summary>Solution: ranking, concentration, and sensitivity</summary>

- **Ranking stays the same.** Dividing by a positive number preserves the order of the scores, so key 3 still receives the largest weight.
- **The distribution becomes softer.** For this example, weights change from approximately $(0.035,0.259,0.705)$ to $(0.074,0.306,0.620)$. Key 3 receives a smaller share, while the other two receive more. With the same values as N1, the output becomes about $(1.55,0.38)$.
- **Unscaled scores become more concentrated as width grows in this experiment.** Their largest weight approaches 1, and the measured sensitivity of $p_0$ to the softmax input tends to fall. Scaled scores keep a more comparable spread across widths. These are averages over random samples, not a guarantee about every query or every training gradient.

Verify the numerical weights with `F.softmax(torch.tensor([1.,3.,4.]) / 2**.5, dim=-1)`.

</details>

## N4 — Multi-Head Attention

**Each head computes its own access distribution; concatenation and projection combine their outputs.** The combined output still has one row per input token.

**Predict:** Use batch size 2, 10 tokens, model width 512, and 8 heads, each with key and value width 64. Fill in the shapes before running:

| Quantity | Shape |
|---|---|
| One head's parameter matrix $W^Q$ | … |
| All attention weights $P$, including batch and head axes | … |
| Combined output after $W^O$ | … |

<details>
<summary>Implementation and width budget</summary>

Slides: “Multi-Head Attention” and “Width of One Head.” A head includes its Q/K/V projections **and** scaled attention. The implementation below uses separate heads to make the formulas readable. It omits projection biases and dropout. `Z` and `mask` support the optional experiments in N8.

The default $d_k=d_v=d/h$ keeps the total projection budget comparable to one full-width head. It is a convention, not a requirement of attention. The implementation requires $d$ divisible by $h$; `Head` itself accepts independent key and value widths.

</details>

In [ ]:
class Head(nn.Module):
    def __init__(self, d, d_k, d_v):
        super().__init__()
        self.W_Q = nn.Parameter(torch.randn(d, d_k) / math.sqrt(d))
        self.W_K = nn.Parameter(torch.randn(d, d_k) / math.sqrt(d))
        self.W_V = nn.Parameter(torch.randn(d, d_v) / math.sqrt(d))
        self.d_k = d_k

    def forward(self, X, Z=None, mask=None):
        Z = X if Z is None else Z                        # self-attention unless Z is given
        Q = X @ self.W_Q                                 # (B, n, d_k)
        K = Z @ self.W_K                                 # (B, m, d_k)
        V = Z @ self.W_V                                 # (B, m, d_v)
        S = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)  # (B, n, m)
        if mask is not None:
            S = S + mask                                 # 0 where allowed, -inf where blocked
        P = F.softmax(S, dim=-1)                         # (B, n, m), each row sums to 1
        return P @ V, P                                  # (B, n, d_v), (B, n, m)


class MultiHeadAttention(nn.Module):
    def __init__(self, d, h):
        super().__init__()
        assert h > 0 and d % h == 0, "choose a positive head count dividing d"
        d_k = d_v = d // h                               # the usual budget convention
        self.heads = nn.ModuleList(Head(d, d_k, d_v) for _ in range(h))
        self.W_O = nn.Parameter(torch.randn(h * d_v, d) / math.sqrt(h * d_v))

    def forward(self, X, Z=None, mask=None):
        results = [head(X, Z, mask) for head in self.heads]
        Y = torch.cat([y for y, _ in results], dim=-1)   # (B, n, h*d_v)
        P = torch.stack([p for _, p in results], dim=1)  # (B, h, n, m)
        return Y @ self.W_O, P                           # (B, n, d), (B, h, n, m)

**Run the shape experiment and compare it with your prediction.**

In [ ]:
torch.manual_seed(0)
d, h, n, B = 512, 8, 10, 2
mha = MultiHeadAttention(d, h)
X_demo = torch.randn(B, n, d)
out, P_demo = mha(X_demo)
print("input            ", X_demo.shape)
print("one head's W_Q   ", mha.heads[0].W_Q.shape)
print("all weights P    ", P_demo.shape)
print("output           ", out.shape)

In [ ]:
def count(module):
    return sum(p.numel() for p in module.parameters())

one_full = Head(512, 512, 512)
eight = MultiHeadAttention(512, 8)
print("one head of width 512, Q+K+V:", count(one_full))
print("eight heads of width 64, Q+K+V:", sum(count(hd) for hd in eight.heads))
print("output projection W_O:", eight.W_O.numel())

**Check yourself.**

1. Check your shape predictions against the printed output. What does each axis represent?
2. The parameter counts show that eight width-64 heads and one width-512 head use the same number of Q/K/V projection parameters. What can the eight heads compute separately that the single head shares across all its value features?
3. Must every head have width $d/h$, or could $W^O$ map a different concatenated width back to $d$?

<details>
<summary>Solution: shapes, independent access distributions, and width choices</summary>

- **Shapes:** one head's $W^Q$ is $(512,64)$, mapping model features to query features. All attention weights have shape $(2,8,10,10)$: batch, head, query position, key position. The combined output is $(2,10,512)$: batch, token position, model feature.
- **Eight heads compute eight distributions per query.** A single head uses one distribution to mix all its value features. Multiple heads can give different positions different weights in different feature subspaces. They are not guaranteed to learn distinct linguistic jobs.
- **The width is a budget choice.** $W^O$ maps the concatenated value width $h d_v$ back to $d$; its shape is $(h d_v,d)$. Key/query width $d_k$ controls the score computation and need not equal $d_v$ or $d/h$. The supplied `MultiHeadAttention` class fixes both widths to $d/h$ for simplicity; trying other budgets requires changing that constructor, not just its input.
- **Compare like with like:** the equal counts are for Q/K/V projections. The output projection is counted separately.

</details>

## N5 — Order Needs a Signal

**Without position information, unrestricted self-attention follows a permutation of its input rows.** Averaging those output rows removes even that permutation.

**Predict:** Reorder the five input rows using `perm`.

1. Will the new output equal the old output with its rows reordered by the same `perm`?
2. Will averaging the output rows give the same vector before and after the reordering?

<details>
<summary>Scope of the experiment</summary>

Slide: “Can Attention Detect Word Order?” This test uses deterministic, unmasked self-attention without positional information. Causal masks and other order-sensitive mechanisms change the assumptions.

</details>

In [ ]:
torch.manual_seed(1)
X_small = torch.randn(1, 5, 512)
perm = torch.tensor([2, 0, 1, 4, 3])

with torch.no_grad():
    out_a, _ = mha(X_small)
    out_b, _ = mha(X_small[:, perm])

print("outputs follow the inputs:", torch.allclose(out_a[:, perm], out_b, atol=1e-5))
print("mean-pooled vectors equal:", torch.allclose(out_a.mean(1), out_b.mean(1), atol=1e-5))

**Position vectors attach token content to sequence slots:** $X=E+E^{\text{pos}}$.

**Predict:** Compare two ways of reordering the input.

1. Move token embeddings to new slots, but leave each slot's position vector in place. Must the outputs still be just a reordering of the old outputs?
2. Move each complete token-plus-position row together. Must the outputs be reordered in exactly the same way?

<details>
<summary>Sinusoidal and learned position vectors</summary>

Slides: “Adding a Position Vector to Each Token” and “Sinusoidal Position Vectors.” A learned table stores a trainable vector for each supported position. The fixed sinusoidal construction below uses

$$E^{\text{pos}}_{p,2m}=\sin(p/10000^{2m/d}),\qquad E^{\text{pos}}_{p,2m+1}=\cos(p/10000^{2m/d}).$$

The plot shows positions as rows and features as columns. Computing a sinusoidal vector beyond the training lengths does not guarantee that a model will generalize to those lengths.

</details>

In [ ]:
def sinusoidal_positions(n, d):
    assert d % 2 == 0, "this formula needs an even width"
    p = torch.arange(n).unsqueeze(1).float()              # (n, 1)
    m = torch.arange(d // 2).float()                      # (d/2,)
    angle = p / 10000 ** (2 * m / d)                      # (n, d/2)
    E = torch.zeros(n, d)
    E[:, 0::2] = torch.sin(angle)
    E[:, 1::2] = torch.cos(angle)
    return E

In [ ]:
E_pos = sinusoidal_positions(50, 64)
plt.figure(figsize=(7, 3.2))
plt.imshow(E_pos, aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
plt.xlabel("embedding dimension"); plt.ylabel("position"); plt.colorbar(); plt.tight_layout(); plt.show()

X_with_pos = X_small + sinusoidal_positions(5, 512)
X_perm_pos = X_small[:, perm] + sinusoidal_positions(5, 512)
with torch.no_grad():
    a, _ = mha(X_with_pos)
    b, _ = mha(X_perm_pos)
    c, _ = mha(X_with_pos[:, perm])
print("reorder tokens; keep slot positions: outputs merely reorder?", torch.allclose(a[:, perm], b, atol=1e-5))
print("reorder complete token+position rows: outputs reorder?", torch.allclose(a[:, perm], c, atol=1e-5))

<details>
<summary>Check the permutation experiments</summary>

- **Without positions:** the output rows follow the same permutation as the input rows. Their mean stays the same because averaging does not depend on row order.
- **Tokens moved between fixed position slots:** the token/position pairings change. The new input is not merely a row permutation of the old input, so the outputs need not be a permutation of the old outputs. The experiment prints `False` for that comparison.
- **Complete token-plus-position rows moved together:** the input really is just a row permutation. Unmasked attention still follows that permutation, and the comparison prints `True`. Position vectors make token-to-slot assignments available to the model; they do not change this mathematical property of attention.

</details>

**Check yourself.** Suppose training sequences contain at most 128 tokens. How would sinusoidal positions and a learned position table handle token 129?

<details>
<summary>Solution: defining a vector does not guarantee useful predictions</summary>

- **Sinusoidal positions:** The fixed formula can compute a vector for position 129 immediately. But the model has never trained on that position, so it may not use the vector effectively.
- **Learned positions:** Each position has its own trainable vector. A table with 128 entries has no vector for position 129 and must be extended. Even if a larger table was allocated, entries never encountered during training have not been learned from the task.
- **The tradeoff:** Sinusoidal positions provide a fixed pattern without learned position parameters. A learned table lets training shape the position vectors for the task, but needs parameters and training coverage for those positions.

</details>

## N6 — The Encoder Block

**Attention exchanges information; the FFN processes each position independently.** Residual addition and LayerNorm wrap each sublayer.

$$U=\operatorname{LN}(X+\operatorname{MHA}(X)),\qquad X'=\operatorname{LN}(U+\operatorname{FFN}(U))$$

**Predict:** Change only the last token. Which token outputs can change after the FFN alone? After attention alone? Record your answers before running the experiment.

| Operation | Only the last output changes, or other positions too? | Why? |
|---|---|---|
| FFN | … | … |
| Attention | … | … |

<details>
<summary>Block variant and LayerNorm axes</summary>

Slides: “The Encoder Block” and “Feed-Forward Network, LayerNorm, and Stacking.” This is a teaching implementation of the original **post-norm** block, with dropout omitted. The FFN shares its parameters across positions.

LayerNorm computes mean and variance across the features of each token independently, then applies learned scale and shift. Its statistics do not aggregate other tokens or batch examples. Its input may nevertheless already contain contextual information from attention. The by-hand calculation uses population variance (`unbiased=False`) and includes epsilon inside the square root.

</details>

In [ ]:
class EncoderBlock(nn.Module):
    def __init__(self, d, h, ff_factor=4):
        super().__init__()
        self.mha = MultiHeadAttention(d, h)
        self.ffn = nn.Sequential(nn.Linear(d, ff_factor * d), nn.ReLU(), nn.Linear(ff_factor * d, d))
        self.ln1 = nn.LayerNorm(d)
        self.ln2 = nn.LayerNorm(d)

    def forward(self, X):
        A, P = self.mha(X)
        U = self.ln1(X + A)
        return self.ln2(U + self.ffn(U)), P

**Run the token intervention.** Compare the two per-token change vectors with your prediction.

In [ ]:
torch.manual_seed(0)
block = EncoderBlock(32, 4)
x = torch.randn(1, 4, 32)
changed = x.clone(); changed[0, 3] += 1.0             # change only the last token
with torch.no_grad():
    ffn_diff = (block.ffn(changed) - block.ffn(x)).abs().sum(dim=-1)
    mha_diff = (block.mha(changed)[0] - block.mha(x)[0]).abs().sum(dim=-1)
print("per-token change after FFN      :", ffn_diff)
print("per-token change after attention:", mha_diff)

<details>
<summary>Optional numerical check: LayerNorm by hand</summary>

The next cell checks normalization across each token’s features against `nn.LayerNorm`, including its learned affine parameters and epsilon.

</details>

In [ ]:
# LayerNorm by hand: statistics over the features of one token
ln = nn.LayerNorm(6)
x = torch.randn(2, 3, 6)                              # (B, n, d)
mu = x.mean(dim=-1, keepdim=True)
var = x.var(dim=-1, keepdim=True, unbiased=False)
by_hand = ln.weight * (x - mu) / torch.sqrt(var + ln.eps) + ln.bias
print("matches nn.LayerNorm:", torch.allclose(by_hand, ln(x), atol=1e-6))

**Check yourself.**

1. After changing only the last token, which output positions change after attention? Which change after the FFN? What does this tell you about information flow?
2. LayerNorm normalizes each token’s feature vector separately. Why might this be useful compared with using statistics from other examples in the batch?

<details>
<summary>Solution: exchanging information versus normalizing a representation</summary>

- **Attention exchanges information between tokens.** In this experiment, changing the last token changes the attention output at every position because each query can access that token’s key and value.
- **The FFN processes tokens independently.** Changing the last token changes only its FFN output. The same FFN parameters are used at every position, but no information passes between positions.
- **LayerNorm normalizes one token’s representation at a time.** Its mean and variance are computed across that token’s features. Its result therefore does not depend on which other examples happen to share the batch, and the same operation works with different batch sizes and sequence lengths. The representation itself may already contain information gathered from other tokens by attention.

</details>

## N7 — Can Training Recover Order? · Synthesis Experiment

**Training cannot recover distinctions that the architecture discards.** Each sequence contains `A` and `B` once, among filler tokens. Predict whether `A` comes before `B`; the classifier averages the encoder's output rows.

**Predict:** Write down an expected test accuracy for each model: no positions, learned positions, and sinusoidal positions. Which model has an architectural reason to stay near 50%, even if it trains longer? For the other two, distinguish “can represent the distinction” from “will successfully learn it in this run.”

<details>
<summary>Experimental setup and dependencies</summary>

Run N4–N6 first. All three models use the same task, model width, head count, block count, training budget, and held-out test set. The position scheme is the intended comparison; learned positions add parameters. One seeded run illustrates the architectural distinction rather than establishing which position scheme trains best. Change seeds before drawing comparative conclusions.

The next three definition cells provide data generation, the classifier, and training. Run them once; the short experiment cell then trains and compares the models. Defaults use 500 updates per model on CPU.

</details>

In [ ]:
A_ID, B_ID, VOCAB = 8, 9, 10

def order_batch(B, n=12, generator=None):
    x = torch.randint(0, 8, (B, n), generator=generator)
    y = torch.randint(0, 2, (B,), generator=generator)        # 1: A before B
    for i in range(B):
        a, b = torch.randperm(n, generator=generator)[:2].sort().values.tolist()
        x[i, a], x[i, b] = (A_ID, B_ID) if y[i] == 1 else (B_ID, A_ID)
    return x, y

In [ ]:
class OrderClassifier(nn.Module):
    def __init__(self, positions, d=32, h=4, N=2, n_max=12):
        super().__init__()
        self.embed = nn.Embedding(VOCAB, d)
        self.positions = positions
        if positions == "learned":
            self.pos = nn.Parameter(torch.randn(n_max, d) * 0.1)
        else:
            self.register_buffer("pos", sinusoidal_positions(n_max, d) if positions == "sinusoidal" else torch.zeros(n_max, d))
        self.blocks = nn.ModuleList(EncoderBlock(d, h) for _ in range(N))
        self.head = nn.Linear(d, 2)

    def forward(self, ids):
        X = self.embed(ids) + self.pos[: ids.shape[1]]        # X = E + E_pos
        for block in self.blocks:
            X, P = block(X)
        return self.head(X.mean(dim=1)), P                    # P of the last block

In [ ]:
def train_order(positions, steps=500, seed=0):
    torch.manual_seed(seed)
    model = OrderClassifier(positions)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    curve = []
    for step in range(steps):
        x, y = order_batch(64)
        loss = F.cross_entropy(model(x)[0], y)
        opt.zero_grad(); loss.backward(); opt.step()
        curve.append(loss.item())
    model.eval()
    g = torch.Generator().manual_seed(123)                    # the same test set for all
    x_test, y_test = order_batch(2000, generator=g)
    with torch.no_grad():
        acc = (model(x_test)[0].argmax(dim=-1) == y_test).float().mean().item()
    return model, curve, acc

**Run the comparison.** Record the three accuracies beside your predictions. Compare training curves, but distinguish optimization behavior from the architectural impossibility without positions.

In [ ]:
results = {}
for positions in ["none", "learned", "sinusoidal"]:
    results[positions] = train_order(positions)
    print(f"{positions:>11}: test accuracy {results[positions][2]:.3f}")

plt.figure(figsize=(6, 3))
for positions, (_, curve, _) in results.items():
    plt.plot(curve, label=positions, alpha=.8)
plt.xlabel("step"); plt.ylabel("training loss"); plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

**Inspect a paired counterexample before interpreting attention maps.** Swap `A` and `B` in one input: its label flips. Without positions, mean pooling should give the same logits for both versions.

<details>
<summary>What an attention heatmap can establish</summary>

The optional heatmaps below show which positions each head accesses for the learned-position model's last block. Rows are queries and columns are keys. They do not by themselves establish which input caused the final prediction. An input intervention and a comparison of predictions provide additional evidence.

</details>

In [ ]:
g = torch.Generator().manual_seed(7)
x_one, y_one = order_batch(1, generator=g)
x_swap = x_one.clone()
x_swap[x_one == A_ID] = B_ID
x_swap[x_one == B_ID] = A_ID
print("input:", x_one.tolist(), "label:", y_one.item())
print("swapped:", x_swap.tolist(), "label:", 1 - y_one.item())
for positions, (model, _, _) in results.items():
    model.eval()
    with torch.no_grad():
        logits, _ = model(x_one)
        swapped_logits, _ = model(x_swap)
    print(positions, "same logits:", torch.allclose(logits, swapped_logits, atol=1e-5),
          "predictions:", logits.argmax(-1).item(), swapped_logits.argmax(-1).item())

<details>
<summary>Optional inspection: learned attention weights</summary>

Run the next cell to inspect the last block’s four heads. Describe an access pattern, then state what additional intervention would be needed to support a claim about its role in the decision.

</details>

In [ ]:
model = results["learned"][0]
g = torch.Generator().manual_seed(7)
x_one, y_one = order_batch(1, generator=g)
with torch.no_grad():
    _, P_one = model(x_one)
names = [{A_ID: "A", B_ID: "B"}.get(t, str(t)) for t in x_one[0].tolist()]

fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for r, ax in enumerate(axes):
    ax.imshow(P_one[0, r], cmap="Blues", vmin=0, vmax=1)
    ax.set_title(f"head {r + 1}")
    ax.set_xticks(range(len(names)), names); ax.set_yticks(range(len(names)), names)
plt.tight_layout(); plt.show()
print("label (1: A before B):", y_one.item())

**Check yourself.**

1. Swap `A` and `B` in a sequence. The correct label flips. Why must the model without positions still produce the same prediction?
2. A head assigns a large attention weight to `A`. Does this establish that `A` determined the model’s prediction?

<details>
<summary>Solution: what the architecture cannot distinguish, and what heatmaps show</summary>

- **Without positions, swapping tokens only reorders the encoder’s output vectors.** The classifier averages those vectors, and their average is unchanged by reordering. It therefore produces the same prediction for both sequences, although their correct labels are opposite.
- **More training cannot remove this limitation.** Each sequence and its swapped counterpart are equally likely in this task. A model that gives both the same prediction cannot systematically beat 50% accuracy. It needs an order-sensitive signal or computation. A finite test set can still give an accuracy slightly above or below 50% by chance.
- **Attention heatmaps show access, not a complete explanation of the prediction.** A large weight means a head gives that position’s value vector a large share in its weighted sum. The final prediction also depends on the value vectors, other heads, and later computations.
- **Test an explanation with an intervention.** Change or remove a token and compare the prediction. This provides additional evidence, although it may change several parts of the computation at once. Patterns may also vary across training runs; try another seed, such as `train_order("learned", seed=1)`.

</details>

## N8 — Three Uses of Attention · Extension

**Query/key/value origins determine what is accessed; masks determine what is allowed.** The N4 implementation supports encoder self-attention, causal self-attention, and cross-attention.

**Predict before running:**

1. Change only target token 4. Under the causal mask, can outputs 1–3 change? Can output 4 change?
2. With 4 target tokens and 7 source tokens, how many rows and columns does one head's cross-attention matrix need?
3. If a weight is set to zero *after* softmax, will the remaining weights still sum to 1?

<details>
<summary>Masking and available context</summary>

Slide: “Three Uses of the Same Attention.” A causal mask keeps the diagonal and earlier positions and blocks later positions by adding $-\infty$ before softmax. Every query needs at least one allowed key; an entirely blocked row is undefined in this implementation.

In standard offline encoder–decoder generation, the entire source is already available. Cross-attention therefore needs no causal source mask, although padded source positions should still be masked. Streaming settings may restrict source availability.

Zeroing weights after softmax without renormalizing does not reproduce masked attention; the comparison below demonstrates the missing normalization.

</details>

In [ ]:
def causal_mask(n):
    # 0 for j <= i (the diagonal stays allowed), -inf for j > i
    return torch.triu(torch.full((n, n), -float("inf")), diagonal=1)

print(causal_mask(4))

torch.manual_seed(0)
mha_small = MultiHeadAttention(16, 2)
tgt = torch.randn(1, 4, 16)
with torch.no_grad():
    y, P = mha_small(tgt, mask=causal_mask(4))
print("weights of head 1:\n", P[0, 0])
print("row sums:", P[0, 0].sum(dim=-1))

# Change the last token. Which outputs may change?
tgt2 = tgt.clone(); tgt2[0, 3] += 1.0
with torch.no_grad():
    y2, _ = mha_small(tgt2, mask=causal_mask(4))
print("change per position:", (y2 - y).abs().sum(dim=-1))

In [ ]:
# Cross-attention: queries from the decoder, keys and values from the encoder
src = torch.randn(1, 7, 16)
with torch.no_grad():
    y_cross, P_cross = mha_small(tgt, Z=src)
print("weights:", P_cross.shape, " output:", y_cross.shape)

# What goes wrong when weights are zeroed after the softmax instead?
S = torch.tensor([[1., 3., 4.]])
after = F.softmax(S, dim=-1).clone(); after[0, 2] = 0.
before = F.softmax(S + torch.tensor([[0., 0., -float("inf")]]), dim=-1)
print("zeroed after softmax:", after, "sum", after.sum().item())
print("-inf before softmax :", before, "sum", before.sum().item())

**Check yourself.**

1. Why does changing target token 4 leave causal-attention outputs 1–3 unchanged in this experiment?
2. Why does cross-attention have a $4\times7$ matrix per head but only 4 output rows?
3. In standard offline translation, why is a causal mask needed for target self-attention but not for access to the source?
4. Why does zeroing a weight after softmax give a different result from masking its score before softmax?

<details>
<summary>Solution: allowed information flow, shapes, and normalization</summary>

- **Earlier queries cannot access the changed token.** For queries 1–3, the mask blocks key/value position 4, and their own queries are unchanged. Their outputs therefore stay the same. Query 4 can access position 4, so its output can change.
- **Each target query scores every available source key.** Four queries and seven keys give a $4\times7$ matrix per head. Each row mixes the seven source values into one output, giving four output rows. Including batch and head axes, the weight tensor is `(1, 2, 4, 7)` in this example.
- **Future targets are unavailable; the source is already known.** Causal target self-attention blocks later target tokens to prevent access to answers not yet generated. In standard offline translation, all source tokens are available before decoding. Source padding may still need masking; streaming translation can impose additional restrictions.
- **Masking before softmax redistributes probability.** A score of $-\infty$ contributes zero to both the numerator and the normalization sum. Zeroing a weight afterward removes its contribution but leaves the other weights unchanged, so they sum to less than 1 in this example. Renormalization would be needed to recover the masked distribution.

</details>

## N9 — Subwords · Extension

**A tokenizer learns a vocabulary and maps text to the IDs expected by its model.** The offline example uses toy BPE: learn frequent adjacent-pair merges, then replay those merges on new text.

**Predict:** Run the training example and write down its three learned merges. Starting from the characters `B P E B P`, apply those merges by hand in the recorded order. What token sequence remains? The solution below lets you check your reasoning.

<details>
<summary>Toy BPE versus the pretrained example</summary>

Slides: “Byte-Pair Encoding Learns Merges” and “Subwords in Practice.” The toy algorithm repeatedly merges a most frequent adjacent pair, breaking ties by first occurrence. It names new symbols `a`, `b`, `c`, …; it is limited to an uppercase toy alphabet and is not a production tokenizer.

The optional BERT example uses **WordPiece**, not BPE. Both create subword vocabularies, but their vocabulary-learning procedures differ. The purpose of that example is to inspect a model-specific tokenizer and its IDs, not to demonstrate the toy merge algorithm.

</details>

In [ ]:
def merge(symbols, pair, new):
    out, i = [], 0
    while i < len(symbols):
        if i + 1 < len(symbols) and (symbols[i], symbols[i + 1]) == pair:   # nonoverlapping, from the left
            out.append(new); i += 2
        else:
            out.append(symbols[i]); i += 1
    return out

def bpe_train(text, n_merges):
    if any(not ("A" <= ch <= "Z") for ch in text):
        raise ValueError("toy training input must use uppercase A–Z; lowercase names are reserved")
    if not 0 <= n_merges <= 26:
        raise ValueError("toy implementation supports 0–26 merges")
    symbols, merges = list(text), []
    for step in range(n_merges):
        if len(symbols) < 2:
            break
        pair, freq = Counter(zip(symbols, symbols[1:])).most_common(1)[0]   # ties: first seen
        new = chr(ord("a") + step)
        merges.append((pair, new))
        symbols = merge(symbols, pair, new)
        print(f"step {step + 1}: {''.join(pair)} -> {new}   {' '.join(symbols)}")
    return merges

def bpe_apply(text, merges):
    symbols = list(text)
    for pair, new in merges:
        symbols = merge(symbols, pair, new)
    return symbols

merges = bpe_train("DBPEBPEBPED", 3)
print("application to BPEBPED:", bpe_apply("BPEBPED", merges))

### Optional: Inspect BERT's WordPiece Tokenizer

Skip the next cell when offline. It downloads tokenizer files, not the model weights. Try a word of your own.

<details>
<summary>Installation and what to inspect</summary>

If needed, run `%pip install transformers` in a separate cell. Compare the readable subwords with the integer IDs and identify the added special tokens. The checkpoint is `bert-base-uncased`, so capitalization is normalized. Its token-to-ID mapping belongs to that checkpoint.

</details>

In [ ]:
try:
    from transformers import AutoTokenizer
except ImportError:
    print("Optional example: install with %pip install transformers, then rerun this cell.")
else:
    try:
        tok = AutoTokenizer.from_pretrained("bert-base-uncased")
    except OSError as e:
        print("Tokenizer files unavailable. Check network access or use cached files.")
        print(f"Download/cache error: {e}")
    else:
        sentence = "The origins of the word octothorpe are shrouded in mystery."
        print(tok.tokenize(sentence))
        ids = tok(sentence)["input_ids"]
        print(ids)
        print(tok.convert_ids_to_tokens(ids))

**Check yourself.**

1. For `BPEBP`, what token sequence remains after each learned merge? Does the third merge apply?
2. Suppose ID 42 means one token in tokenizer A and a different token in tokenizer B. What happens if a model trained with A receives B's ID 42?

<details>
<summary>Solution: replaying merges and looking up the wrong embedding</summary>

- **First merge:** `BP → a` changes `B P E B P` into `a E a`.
- **Second merge:** `aE → b` changes `a E a` into `b a`.
- **Third merge:** `bb → c` does not apply because there are no adjacent `b b` tokens. The final token sequence is `['b', 'a']`. Application reuses learned merges; it does not learn new ones for this input.
- **An ID selects a row in the model's embedding matrix.** The model interprets 42 according to tokenizer A, regardless of which tokenizer produced it. If B assigns 42 a different meaning, the model receives the embedding for the wrong token. This may produce no error when the ID is in range, but the input is misrepresented. Use the tokenizer supplied for the model checkpoint.

</details>

## Put the Encoder Together

Answer without opening the explanations:

1. Changing one token can affect outputs at other positions. Which sublayer allows this? Which sublayer transforms each position independently?
2. Why does averaging encoder outputs make `A`-before-`B` impossible to classify without an order-sensitive signal? How do position vectors address this limitation?
3. At ordinary inference, do projection matrices change when a new sentence arrives? Do attention weights change? Explain the difference.

<details>
<summary>Check your synthesis</summary>

**1. Information flow:** Attention mixes values from allowed positions. The FFN transforms each position separately with shared parameters; LayerNorm also normalizes each token separately. Residual additions combine the current representation with sublayer updates.

**2. Order:** Without an order-sensitive signal, the unmasked encoder follows row permutations and mean pooling makes them indistinguishable. Position vectors attach content to slots.

**3. Fixed parameters, changing computations:** At inference, learned projection matrices, FFN parameters, embeddings, and LayerNorm scale/shift remain fixed. Token representations, Q/K/V, scores, attention weights, and outputs are computed from the current input. Input-dependent attention weights are not new training updates.

</details>

## Going Further

Choose one investigation. Predict first, keep the comparison controlled, and report what your evidence supports.

- **Depth:** Train models with 1, 2, and 4 blocks by changing `N` in `OrderClassifier`. Keep the position scheme, width, training steps, and test set fixed. Compare test accuracy and elapsed training time over several seeds. Does extra depth help on this task, and at what cost?
- **Task:** Replace the label with “are `A` and `B` at most 3 positions apart?” Generate labels from the sampled positions and check the positive/negative class balance. Compare learned and sinusoidal positions with the same data and training budget over several seeds. Which learns faster in your runs? Report the majority-class baseline as well as accuracy.
- **Pre-norm:** Rewrite `EncoderBlock.forward` as $U=X+\operatorname{MHA}(\operatorname{LN}(X))$, $X'=U+\operatorname{FFN}(\operatorname{LN}(U))$. Which operations now receive normalized inputs? Which path carries the residual representation without passing through LayerNorm?

<details>
<summary>Check the pre-norm computation graph</summary>

MHA receives `ln1(X)` and the FFN receives `ln2(U)`. The residual additions carry `X` into `U` and `U` into `X'` directly. In the post-norm block, LayerNorm instead follows each residual addition. Keep the MHA return value unpacking (`A, P`) when rewriting the code; the attention module returns both the update and the weights.

</details>